# Session 2.2 -- Chunking and Vector Ingestion

**AI-2: AI Backend Engineering**

---

## Learning Objectives

By the end of this session, you will be able to:

1. **Compare** fixed-size, overlap, and paragraph-aware chunking strategies and select one appropriate for a given document type
2. **Explore** the pre-built chunking function, experiment with parameters that split documents and attach structured metadata
3. **Ingest** chunked and embedded documents into ChromaDB and verify retrieval with a similarity query

## What You Are Working With

- `src/s3_ingestion/chunker.py` -- Chunking module with `Chunk` dataclass and `chunk_document()` / `chunk_fixed()` (provided complete)
- `src/s3_ingestion/store.py` -- ChromaDB storage with `ingest_chunks()`, `verify_ingestion()`, `reset_collection()` (provided complete)
- `src/s2_embeddings/embed.py` -- Voyage AI embedding functions from Session 2.1 (provided complete)
- `data/northbrook/` -- Corpus of memos, meeting notes, financial reports, and policies from Northbrook Partners

You **import and use** the pre-built modules. You do not need to build them from scratch.

---

## 1. Setup and Imports

In [ ]:
import sys
from pathlib import Path

# Find repo root by walking up until we find pyproject.toml
_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))

from dotenv import load_dotenv
load_dotenv()

In [ ]:
import os
from pathlib import Path
import textwrap

# Pre-built chunking module
from src.s3_ingestion.chunker import chunk_document, chunk_fixed, Chunk, count_tokens

# Pre-built storage module
from src.s3_ingestion.store import get_collection, ingest_chunks, verify_ingestion, reset_collection

# Embedding module from Session 2.1
from src.s2_embeddings.embed import embed_texts, cosine_similarity

print("All imports loaded successfully.")

In [ ]:
# Reset the collection so this notebook is idempotent.
# Running this cell guarantees a clean ChromaDB state every time.
reset_collection()
print("Collection reset. Starting fresh.")

---

## 2. Why Chunking Matters

In Session 2.1, we embedded short text snippets and measured similarity. In the real world, documents are long -- a policy might be 750 tokens, a handbook might be 1,250. You cannot embed the whole thing as one vector and expect good retrieval.

Let's see the problem firsthand.

In [ ]:
# Load a full Northbrook document
doc_path = _root / "data" / "northbrook" / "employee_handbook.md"
handbook_text = doc_path.read_text()

print(f"Document: {doc_path.name}")
print(f"Length:   {len(handbook_text):,} characters ({count_tokens(handbook_text):,} tokens)")
print(f"Lines:    {len(handbook_text.splitlines())}")
print()
print("--- First 300 characters ---")
print(handbook_text[:300])
print("...")

In [ ]:
# This document covers many different topics. Let's see what sections it contains.
sections = [line for line in handbook_text.splitlines() if line.startswith("## ")]
print(f"Sections in the handbook ({len(sections)} total):")
for s in sections:
    print(f"  {s}")

In [ ]:
# The problem: if we embed this entire document as ONE vector, that single
# vector has to represent Dress Code AND Performance Reviews AND Onboarding
# AND Professional Development all at once.
#
# Let's prove it. Embed the whole document, then compare it to specific questions.

full_doc_embedding = embed_texts([handbook_text])[0]

questions = [
    "What is the dress code at Northbrook Partners?",
    "How does the performance review process work?",
    "What is the professional development budget?",
    "What is the Navigator onboarding program?",
]

question_embeddings = embed_texts(questions)

print("Similarity between FULL DOCUMENT embedding and each question:\n")
for q, q_emb in zip(questions, question_embeddings):
    sim = cosine_similarity(full_doc_embedding, q_emb)
    print(f"  {sim:.4f}  {q}")

print("\nNotice: all scores are moderate. The single embedding tries to represent")
print("everything and ends up representing nothing well.")

**The tradeoff at the heart of chunking:**

- **Too big** = diluted meaning. A single vector cannot represent five different topics well.
- **Too small** = lost context. A sentence fragment like "the policy applies" means nothing on its own.

Every chunking strategy is a different answer to this tension.

---

## 3. Exploring `chunk_document()`

The pre-built module `src/s3_ingestion/chunker.py` provides two functions:

- **`chunk_document()`** -- paragraph-aware chunking (merges small paragraphs, respects boundaries)
- **`chunk_fixed()`** -- simple fixed-size chunking (every N tokens with overlap)

Both return `Chunk` objects with `.text` and `.metadata`.

Let's start by exploring `chunk_document()` with different parameters.

In [ ]:
# Load the vacation policy -- a well-structured document with clear sections
vacation_text = (_root / "data" / "northbrook" / "vacation_policy_2025.md").read_text()

print(f"Document: vacation_policy_2025.md")
print(f"Length:   {len(vacation_text):,} characters ({count_tokens(vacation_text):,} tokens)")
print()

# Chunk it with default parameters
chunks = chunk_document(vacation_text, source="vacation_policy_2025.md", doc_type="policy")

print(f"Chunks created: {len(chunks)}")
print(f"Average chunk size: {sum(c.metadata['token_count'] for c in chunks) / len(chunks):.0f} tokens")
print()

# Preview each chunk
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i} ({chunk.metadata['token_count']} tokens) ---")
    print(chunk.text[:150].replace("\n", " "))
    print("...")
    print()

In [ ]:
# Now let's vary the chunk_size and see the impact
print(f"{'chunk_size':>12}  {'overlap':>8}  {'chunks':>8}  {'avg_size':>10}  {'min_size':>10}  {'max_size':>10}")
print("-" * 70)

for size in [50, 75, 128, 200, 256, 400]:
    overlap = size // 5  # 20% overlap
    test_chunks = chunk_document(
        vacation_text,
        source="vacation_policy_2025.md",
        chunk_size=size,
        overlap=overlap,
    )
    sizes = [c.metadata['token_count'] for c in test_chunks]
    avg = sum(sizes) / len(sizes)
    print(f"{size:>12}  {overlap:>8}  {len(test_chunks):>8}  {avg:>10.0f}  {min(sizes):>10}  {max(sizes):>10}")

print()
print("Smaller chunk_size = more chunks = more embeddings = higher cost")
print("Larger chunk_size  = fewer chunks = less cost = but less precise retrieval")

In [ ]:
# Interactive exploration: vary chunk_size and overlap to feel the tradeoff.
# If ipywidgets is available, this creates interactive sliders.
# If not, we fall back to a static comparison.

try:
    import ipywidgets as widgets
    from IPython.display import display, HTML

    def explore_chunking(chunk_size=128, overlap=25):
        test_chunks = chunk_document(
            vacation_text,
            source="vacation_policy_2025.md",
            chunk_size=chunk_size,
            overlap=overlap,
        )
        sizes = [c.metadata['token_count'] for c in test_chunks]
        avg_size = sum(sizes) / len(sizes) if sizes else 0

        # Estimated embedding cost (Voyage AI voyage-3-lite: ~$0.00006 per 1K tokens)
        total_tokens = sum(sizes)
        est_cost = total_tokens / 1000 * 0.00006

        print(f"chunk_size={chunk_size}, overlap={overlap} ({overlap/chunk_size*100:.0f}%)")
        print(f"  Chunks: {len(test_chunks)}")
        print(f"  Avg size: {avg_size:.0f} tokens | Min: {min(sizes)} | Max: {max(sizes)}")
        print(f"  Total tokens: {total_tokens:,}")
        print(f"  Est. embedding cost: ${est_cost:.6f}")

        # Filmstrip: show first 5 chunks with boundaries
        print(f"\n  --- First {min(5, len(test_chunks))} chunks (filmstrip) ---")
        for i, chunk in enumerate(test_chunks[:5]):
            preview = chunk.text[:80].replace("\n", " ")
            print(f"  [{i}] ({chunk.metadata['token_count']:>4} tokens) {preview}...")

    size_slider = widgets.IntSlider(value=128, min=50, max=300, step=25, description="chunk_size (tokens)")
    overlap_slider = widgets.IntSlider(value=25, min=0, max=75, step=5, description="overlap (tokens)")

    widgets.interact(explore_chunking, chunk_size=size_slider, overlap=overlap_slider)

except ImportError:
    print("ipywidgets not available -- showing static comparison instead.")
    print("Install with: pip install ipywidgets\n")

    for size, overlap in [(50, 0), (128, 25), (256, 50)]:
        test_chunks = chunk_document(
            vacation_text,
            source="vacation_policy_2025.md",
            chunk_size=size,
            overlap=overlap,
        )
        sizes = [c.metadata['token_count'] for c in test_chunks]
        print(f"chunk_size={size}, overlap={overlap}: {len(test_chunks)} chunks, avg {sum(sizes)//len(sizes)} tokens")
        for i, c in enumerate(test_chunks[:3]):
            print(f"  [{i}] {c.text[:60].replace(chr(10), ' ')}...")
        print()

---

## 4. Chunk Destruction Demo

What happens when chunking splits a critical detail across two chunks? The answer straddles the boundary and **neither chunk** contains the full information.

This is the seam problem.

In [ ]:
# Take a specific paragraph from the vacation policy that contains a complete thought
# with conditions: the Recharge Days section.
# This paragraph contains: what it is, how many, how to use, and the carryover rule.

recharge_section = """Northbrook Partners is introducing Recharge Days as a new benefit for 2025. In addition to standard PTO, all employees receive 2 Recharge Days per calendar year. Recharge Days are company-encouraged mental health days that require no justification or advance notice. Employees simply notify their manager that they are taking a Recharge Day. Recharge Days do not carry over and must be used within the calendar year. The intent of Recharge Days is to normalize proactive rest. We encourage all employees to take advantage of this benefit."""

print("=== FULL PARAGRAPH (coherent) ===")
print()
print(textwrap.fill(recharge_section, width=80))
print(f"\nLength: {len(recharge_section)} characters ({count_tokens(recharge_section)} tokens)")

In [ ]:
# Now chunk it with fixed-size, NO overlap -- watch the destruction
fixed_no_overlap = chunk_fixed(
    recharge_section,
    source="demo",
    chunk_size=64,
    overlap=0,
)

print(f"Fixed chunking (64 tokens, NO overlap) --> {len(fixed_no_overlap)} chunks\n")

for i, chunk in enumerate(fixed_no_overlap):
    print(f"--- Chunk {i} ({chunk.metadata['token_count']} tokens) ---")
    print(chunk.text)
    if i < len(fixed_no_overlap) - 1:
        # Show the cut point
        end_chars = chunk.text[-30:]
        next_start = fixed_no_overlap[i + 1].text[:30]
        print(f"\n  >>> CUT POINT <<<")
        print(f"  Chunk {i} ends with:    ...{end_chars}")
        print(f"  Chunk {i+1} starts with: {next_start}...")
    print()

In [ ]:
# Now with overlap -- see how it recovers some context at the seam
fixed_with_overlap = chunk_fixed(
    recharge_section,
    source="demo",
    chunk_size=64,
    overlap=12,
)

print(f"Fixed chunking (64 tokens, 12 overlap) --> {len(fixed_with_overlap)} chunks\n")

for i, chunk in enumerate(fixed_with_overlap):
    print(f"--- Chunk {i} ({chunk.metadata['token_count']} tokens) ---")
    print(chunk.text)
    print()

print("Overlap recovers some context at the boundaries, but it is not free --")
print(f"we went from {len(fixed_no_overlap)} chunks to {len(fixed_with_overlap)} chunks.")
print("More chunks = more embeddings = more cost.")

In [ ]:
# Compare: paragraph-aware chunking on the same text keeps it whole
para_chunks = chunk_document(
    recharge_section,
    source="demo",
    chunk_size=150,
)

print(f"Paragraph-aware chunking (150 tokens) --> {len(para_chunks)} chunk(s)\n")
for i, chunk in enumerate(para_chunks):
    print(f"--- Chunk {i} ({chunk.metadata['token_count']} tokens) ---")
    print(textwrap.fill(chunk.text, width=80))
    print()

print("The paragraph-aware strategy kept the complete thought together.")
print("No cut, no lost context, no dangling fragments.")

**Takeaway:** The seam problem is real. When chunking splits a critical detail, neither chunk tells the full story. Overlap helps, but paragraph-aware chunking avoids the problem entirely for well-structured documents.

---

## 5. Chunk Autopsy

Every `Chunk` object has two fields:
- `.text` -- the chunk content
- `.metadata` -- a dictionary with information about where this chunk came from

Metadata is what lets us filter and trace results back to their source. Let's inspect a real chunk in detail.

In [ ]:
# Chunk the vacation policy and inspect a single chunk in detail
chunks = chunk_document(
    vacation_text,
    source="vacation_policy_2025.md",
    doc_type="policy",
)

# Pick chunk 2 (index 2) as our specimen
specimen = chunks[2]

print("=== Chunk Autopsy ===")
print(f"\nChunk index: 2 of {len(chunks)} total")
print(f"Text length: {len(specimen.text)} characters ({count_tokens(specimen.text)} tokens)")
print()
print("--- Metadata ---")
for key, value in specimen.metadata.items():
    print(f"  {key:15s} = {value!r}")
print()
print("--- Full Text ---")
print(specimen.text)

In [ ]:
# Metadata is what ChromaDB stores alongside the embedding.
# When you retrieve a chunk, metadata tells you:
#   - source: which document it came from
#   - chunk_index: where in the document it appeared
#   - doc_type: the category of document
#
# In Week 3, metadata will enable FILTERED retrieval:
#   "Find chunks about vacation, but ONLY from policy documents"

print("Metadata for all chunks in this document:\n")
print(f"{'Index':>6}  {'Source':<30}  {'doc_type':<10}  {'chunk_index':<12}  {'Tokens':>6}")
print("-" * 75)
for chunk in chunks:
    m = chunk.metadata
    print(f"{m.get('chunk_index', '?'):>6}  {m.get('source', '?'):<30}  {m.get('doc_type', '?'):<10}  {m.get('chunk_index', '?'):<12}  {m.get('token_count', count_tokens(chunk.text)):>6} tokens")

---

## 6. Fixed vs Paragraph-Aware Chunking

The module provides two strategies:
- **`chunk_fixed()`** -- cuts every N characters, ignoring content boundaries
- **`chunk_document()`** -- merges paragraphs up to a size limit, respects natural breaks

Let's run both on the same document and compare.

In [ ]:
# Load the remote work policy -- another structured document
remote_text = (_root / "data" / "northbrook" / "remote_work_policy.md").read_text()

# Fixed-size chunking
fixed_chunks = chunk_fixed(
    remote_text,
    source="remote_work_policy.md",
    chunk_size=128,
    overlap=25,
    doc_type="policy",
)

# Paragraph-aware chunking
para_chunks = chunk_document(
    remote_text,
    source="remote_work_policy.md",
    chunk_size=128,
    overlap=25,
    doc_type="policy",
)

print(f"{'':25s}  {'Fixed':>10}  {'Paragraph':>12}")
print("-" * 50)

fixed_sizes = [c.metadata['token_count'] for c in fixed_chunks]
para_sizes = [c.metadata['token_count'] for c in para_chunks]

print(f"{'Chunk count':25s}  {len(fixed_chunks):>10}  {len(para_chunks):>12}")
print(f"{'Average size (tokens)':25s}  {sum(fixed_sizes)//len(fixed_sizes):>10}  {sum(para_sizes)//len(para_sizes):>12}")
print(f"{'Min size':25s}  {min(fixed_sizes):>10}  {min(para_sizes):>12}")
print(f"{'Max size':25s}  {max(fixed_sizes):>10}  {max(para_sizes):>12}")
print(f"{'Total tokens':25s}  {sum(fixed_sizes):>10}  {sum(para_sizes):>12}")

In [ ]:
# Side-by-side: first 3 chunks from each strategy
print("=" * 80)
print("FIXED-SIZE CHUNKS")
print("=" * 80)
for i, chunk in enumerate(fixed_chunks[:3]):
    print(f"\n--- Fixed Chunk {i} ({chunk.metadata['token_count']} tokens) ---")
    print(textwrap.fill(chunk.text[:200], width=80))
    if len(chunk.text) > 200:
        print("...")

print()
print("=" * 80)
print("PARAGRAPH-AWARE CHUNKS")
print("=" * 80)
for i, chunk in enumerate(para_chunks[:3]):
    print(f"\n--- Paragraph Chunk {i} ({chunk.metadata['token_count']} tokens) ---")
    print(textwrap.fill(chunk.text[:200], width=80))
    if len(chunk.text) > 200:
        print("...")

print()
print("Notice the difference: fixed chunks cut wherever the token count lands.")
print("Paragraph chunks respect natural boundaries between topics.")

In [ ]:
# Boundary quality comparison
def count_clean_boundaries(chunks_list):
    """Count how many chunk boundaries end at a natural break."""
    clean = 0
    total = len(chunks_list) - 1
    for i in range(total):
        if chunks_list[i].text.rstrip().endswith(('.', '!', '?', ':', '\n')):
            clean += 1
    return clean, total

fixed_clean, fixed_total = count_clean_boundaries(fixed_chunks)
para_clean, para_total = count_clean_boundaries(para_chunks)

print("Boundary Quality:\n")
print(f"  Fixed:     {fixed_clean}/{fixed_total} boundaries end at a natural break ({fixed_clean/max(fixed_total,1)*100:.0f}%)")
print(f"  Paragraph: {para_clean}/{para_total} boundaries end at a natural break ({para_clean/max(para_total,1)*100:.0f}%)")

**Key insight:** For structured documents with clear sections and paragraphs (like the Northbrook policies), paragraph-aware chunking produces higher quality chunks. For unstructured content without paragraph breaks (like raw logs), fixed-size may be the only option.

Neither strategy is universally "best." The right choice depends on your documents.

---

## 7. Vector Store Ingestion

Now that we can chunk documents, we need to store them in a place where we can search by meaning. That place is **ChromaDB** -- an open-source vector database.

The pipeline is:

```
Document  -->  chunk_document()  -->  Chunk[]  -->  ingest_chunks()  -->  ChromaDB
              (text + metadata)                   (embed + store)
```

The `ingest_chunks()` function from `store.py` handles the embed-and-store step. Let's walk through it.

In [ ]:
# First, let's understand what ingest_chunks() does by looking at its behavior.
# It takes a list of Chunk objects and:
#   1. Extracts the text from each chunk
#   2. Calls embed_texts() (Voyage AI) in batches of 50
#   3. Generates a deterministic ID for each chunk: "{source}_chunk{index}"
#   4. Adds everything to a ChromaDB collection

# Let's ingest the vacation policy chunks
vacation_chunks = chunk_document(
    vacation_text,
    source="vacation_policy_2025.md",
    doc_type="policy",
)

print(f"Ingesting {len(vacation_chunks)} chunks from vacation_policy_2025.md...\n")

result = ingest_chunks(vacation_chunks)
print(f"\nResult: {result}")

In [ ]:
# Why deterministic IDs matter:
# The ID for each chunk is constructed from its source and index.
# This means if you run ingestion twice, ChromaDB rejects duplicates
# instead of creating copies. That is idempotency.

print("Generated IDs for the vacation policy chunks:\n")
for chunk in vacation_chunks:
    chunk_id = f"{chunk.metadata['source']}_chunk{chunk.metadata['chunk_index']}"
    print(f"  {chunk_id}")

In [ ]:
# Let's check the collection state
collection = get_collection()
print(f"Collection: {collection.name}")
print(f"Total documents stored: {collection.count()}")

In [ ]:
# Now ingest the FULL Northbrook corpus
data_dir = _root / "data" / "northbrook"
all_files = sorted(data_dir.glob("*.md"))

print(f"Found {len(all_files)} documents in the Northbrook corpus:\n")
for f in all_files:
    print(f"  {f.name} ({count_tokens(f.read_text()):,} tokens)")

### Classify and Ingest the Full Corpus

Now that we can chunk documents, we need to store them somewhere searchable. The pipeline:

1. **Read** each document from `data/northbrook/`
2. **Classify** its document type (memo, policy, financial, meeting) — this metadata enables filtered retrieval in Week 3
3. **Chunk** it using `chunk_document()` with our default parameters
4. **Ingest** the chunks into ChromaDB via `ingest_chunks()`

The `classify_doc_type()` helper below maps filenames to categories. Run the next cell to ingest the full corpus.

In [ ]:
# Reset the collection so we start clean for the full corpus ingestion
reset_collection()
print("Collection reset for full corpus ingestion.\n")


def classify_doc_type(filename: str) -> str:
    """Classify a Northbrook document by its filename."""
    name = filename.lower()
    if "memo" in name:
        return "memo"
    elif "policy" in name or "handbook" in name:
        return "policy"
    elif "financial" in name:
        return "financial"
    elif "meeting" in name or "standup" in name or "committee" in name:
        return "meeting"
    return "unknown"


# Ingest the full corpus
total_chunks = 0

for doc_path in all_files:
    text = doc_path.read_text()
    doc_type = classify_doc_type(doc_path.name)
    chunks = chunk_document(text, source=doc_path.name, doc_type=doc_type)
    result = ingest_chunks(chunks)
    total_chunks += len(chunks)
    print(f"  {doc_path.name:<40} type={doc_type:<10} chunks={len(chunks)}")

print(f"\n{'=' * 60}")
print(f"Total documents processed: {len(all_files)}")
print(f"Total chunks ingested:     {total_chunks}")

In [ ]:
# Verify the collection is populated
collection = get_collection()
print(f"Collection '{collection.name}' now contains {collection.count()} chunks.")
print(f"\nThis collection persists to disk in chroma_db/ at the repo root.")
print("It will survive between sessions -- you will use it in Week 3 for RAG.")

---

## 8. Verification

The `verify_ingestion()` function embeds a query, searches the collection, and returns the most similar chunks. This is the retrieval step that will form the foundation of RAG next week.

Let's test it.

In [ ]:
# Run a verification query
results = verify_ingestion("What is the vacation policy at Northbrook Partners?", n_results=5)

print("Query: 'What is the vacation policy at Northbrook Partners?'\n")

for i in range(len(results["ids"][0])):
    doc_id = results["ids"][0][i]
    doc_text = results["documents"][0][i]
    metadata = results["metadatas"][0][i]
    distance = results["distances"][0][i]
    similarity = 1 - distance  # ChromaDB cosine distance = 1 - similarity

    print(f"[{i+1}] Score: {similarity:.4f}  |  Source: {metadata.get('source', '?')}  |  Type: {metadata.get('doc_type', '?')}")
    print(f"    Chunk index: {metadata.get('chunk_index', '?')}")
    print(f"    {doc_text[:150].replace(chr(10), ' ')}...")
    print()

In [ ]:
# Try several different queries to see how the retrieval works
test_queries = [
    "What is the remote work policy?",
    "How much is the professional development budget?",
    "What was discussed at the board meeting?",
    "What are the security update details?",
    "How does the Navigator onboarding program work?",
]

for query in test_queries:
    results = verify_ingestion(query, n_results=2)
    print(f"Q: {query}")
    for i in range(len(results["ids"][0])):
        metadata = results["metadatas"][0][i]
        distance = results["distances"][0][i]
        similarity = 1 - distance
        text_preview = results["documents"][0][i][:80].replace("\n", " ")
        print(f"  [{similarity:.4f}] {metadata.get('source', '?'):40s} {text_preview}...")
    print()

**What to notice:**

- The scores tell you how similar the retrieved chunk is to your query (higher = more similar)
- The metadata tells you which document and section the chunk came from
- Different queries surface chunks from different documents -- the retrieval is working
- Some results may surprise you -- semantic similarity is not the same as keyword matching

---

## 9. The Chunking Workbench

Time to get your hands dirty. You have a populated vector store — now let's prove that chunking parameters **directly change answer quality**.

**The setup:**
- 5 documents from the Northbrook corpus
- 1 fixed question: *"What are the approved Recharge Days and how do they work?"*
- You control: `chunk_size`, `overlap`, and `top_k`
- Each run: chunks the docs → ingests → retrieves → sends to Claude → scores the answer

**The scoring:** Your answer is checked for 4 key facts. You get a score from 0–100 that rewards both **correctness** (did Claude get the facts right?) and **efficiency** (did you use fewer chunks and tokens?).

The brute-force ceiling is 60. Finding the right chunks earns 90+.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# CHUNKING WORKBENCH — Setup
# ═══════════════════════════════════════════════════════════════════════

import json
import chromadb
from pathlib import Path
from datetime import datetime, timezone
from src.s0_generation.generate import call_claude_with_usage
from src.s3_ingestion.store import CHROMA_PATH

WORKBENCH_COLLECTION = "workbench_experiment"
WORKBENCH_QUESTION = "What are the approved Recharge Days and how do they work?"
WORKBENCH_JSONL = Path("workbench_runs.jsonl")

# The 5 experiment documents
WORKBENCH_DOCS = [
    "vacation_policy_2025.md",
    "hr_committee_2025_01.md",
    "memo_policy_correction.md",
    "vacation_policy_2023.md",
    "employee_handbook.md",
]

# 4 facts we check for in Claude's answer
FACT_CHECKS = {
    "How many days?": ["2 recharge", "two recharge", "2 days"],
    "No justification needed?": [
        "no justification", "no advance notice", "simply notify",
        "without justification", "without advance notice",
    ],
    "No carryover?": [
        "do not carry over", "don't carry over", "must be used",
        "cannot carry", "use within",
    ],
    "Mental health purpose?": [
        "mental health", "proactive rest", "well-being", "wellness",
    ],
}

WORKBENCH_SYSTEM_PROMPT = (
    "You are a Q&A assistant for Northbrook Partners. "
    "Answer using ONLY the provided context. "
    "Cite your sources by name. "
    "If the context is insufficient, say: "
    "'I don't have enough information to answer that question.'"
)


def _check_facts(answer: str) -> dict[str, bool]:
    """Check which key facts appear in the answer."""
    lower = answer.lower()
    return {name: any(kw in lower for kw in kws) for name, kws in FACT_CHECKS.items()}


def _score_run(facts_hit: int, num_chunks: int, total_tokens: int) -> dict:
    """Score: correctness (0-60) × efficiency multiplier (1.0-1.667)."""
    correctness = (facts_hit / 4) * 60
    token_bonus = max(0.0, 1.0 - (total_tokens / 4000)) * 0.4
    chunk_bonus = max(0.0, 1.0 - (num_chunks / 8)) * 0.267
    mult = 1.0 + min(0.667, token_bonus + chunk_bonus)
    return {"correctness": round(correctness, 1), "multiplier": round(mult, 3),
            "final": min(100, round(correctness * mult))}


def run_workbench(chunk_size: int, overlap: int, top_k: int):
    """Run the full workbench experiment with your parameters."""
    data_dir = _root / "data" / "northbrook"

    # 1. Chunk the experiment documents
    all_chunks = []
    for fname in WORKBENCH_DOCS:
        text = (data_dir / fname).read_text()
        all_chunks.extend(chunk_document(text, source=fname, chunk_size=chunk_size, overlap=overlap))

    # 2. Reset and ingest into experiment collection
    client = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        client.delete_collection(WORKBENCH_COLLECTION)
    except Exception:
        pass
    ingest_chunks(all_chunks, collection_name=WORKBENCH_COLLECTION)

    # 3. Retrieve
    results = verify_ingestion(WORKBENCH_QUESTION, n_results=top_k, collection_name=WORKBENCH_COLLECTION)
    distances = results["distances"][0]
    retrieved = []
    for i, d in enumerate(distances):
        retrieved.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "score": round(1 - d, 4),
        })

    # 4. Build prompt and call Claude
    if not retrieved:
        answer, input_tokens = "No relevant documents found.", 0
    else:
        blocks = []
        for c in retrieved:
            src = c["metadata"].get("source", "Unknown")
            blocks.append(f"[Source: {src}, Score: {c['score']:.3f}]\n{c['text']}")
        context = "\n\n---\n\n".join(blocks)
        user_msg = f"Context:\n\n{context}\n\n---\n\nQuestion: {WORKBENCH_QUESTION}"
        result = call_claude_with_usage(
            prompt=user_msg, system_prompt=WORKBENCH_SYSTEM_PROMPT, temperature=0.0,
            model="claude-haiku-4-5",
        )
        answer, input_tokens = result["text"], result["input_tokens"]

    # 5. Score
    facts = _check_facts(answer)
    hits = sum(facts.values())
    score = _score_run(hits, len(retrieved), input_tokens)

    # 6. Display results
    print(f"{'='*60}")
    print(f"  SCORE: {score['final']}/100")
    print(f"  Correctness: {score['correctness']}/60 | Efficiency: ×{score['multiplier']}")
    print(f"{'='*60}")
    print(f"\n  Parameters: chunk_size={chunk_size}, overlap={overlap}, top_k={top_k}")
    print(f"  Chunks in collection: {len(all_chunks)} | Retrieved: {len(retrieved)}")
    print(f"  Tokens sent to Claude: {input_tokens}")
    print(f"\n  Fact Check:")
    for fact_name, hit in facts.items():
        print(f"    {'✓' if hit else '✗'} {fact_name}")

    print(f"\n  Retrieved chunks:")
    for i, c in enumerate(retrieved):
        src = c["metadata"].get("source", "?")
        print(f"    [{i+1}] {src} (score: {c['score']:.3f})")
        print(f"        {c['text'][:120]}...")

    print(f"\n  Claude's answer:")
    print(f"  {answer[:500]}")
    if len(answer) > 500:
        print("  ...")

    # 7. Log to JSONL
    run_record = {
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "chunk_size": chunk_size, "overlap": overlap, "top_k": top_k,
        "total_chunks": len(all_chunks), "chunks_retrieved": len(retrieved),
        "input_tokens": input_tokens, "facts": facts, "facts_hit": hits,
        "score": score["final"],
    }
    with open(WORKBENCH_JSONL, "a") as f:
        f.write(json.dumps(run_record) + "\n")

    # Cleanup
    try:
        client.delete_collection(WORKBENCH_COLLECTION)
    except Exception:
        pass

    return run_record


print("Workbench ready. Run the next cell to experiment!")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# YOUR TURN: Tune these parameters and re-run this cell!
# ═══════════════════════════════════════════════════════════════════════

# Change these values and re-run (Shift+Enter) to see how the score changes
chunk_size = 50        # Try: 50, 100, 128, 200, 256
overlap    = 0         # Try: 0, 10, 25, 50
top_k      = 3         # Try: 3, 5, 8

run_workbench(chunk_size, overlap, top_k)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# LEADERBOARD — See your progression across runs
# ═══════════════════════════════════════════════════════════════════════

if WORKBENCH_JSONL.exists():
    runs = [json.loads(line) for line in open(WORKBENCH_JSONL) if line.strip()]
    runs.sort(key=lambda r: r["score"], reverse=True)

    print(f"{'='*72}")
    print(f"  LEADERBOARD — {len(runs)} runs")
    print(f"{'='*72}")
    print(f"  {'#':<4} {'Score':<7} {'Facts':<7} {'Chunks':<8} {'Tokens':<8} "
          f"{'Size':<6} {'Overlap':<8} {'TopK':<5}")
    print(f"  {'-'*68}")
    for i, r in enumerate(runs):
        print(f"  {i+1:<4} {r['score']:<7} {r['facts_hit']}/4{'':<4} "
              f"{r['chunks_retrieved']:<8} {r['input_tokens']:<8} "
              f"{r['chunk_size']:<6} {r['overlap']:<8} {r['top_k']:<5}")
    print(f"  {'='*68}")

    best = runs[0]
    print(f"\n  Best run: score {best['score']} with "
          f"size={best['chunk_size']}, overlap={best['overlap']}, top_k={best['top_k']}")
else:
    print("No runs yet. Run the cell above first!")

### What did you discover?

Some patterns you may have noticed:
- **Small chunks (50 tokens) + low top_k (3)** → Claude says "I don't have enough information." The Recharge Days section header ends up in one chunk, but the details (2 days, no justification, no carryover) land in a different chunk that doesn't get retrieved.
- **Adding overlap** partially recovers — you might go from 1/4 to 2/4 facts.
- **Increasing top_k** also helps — retrieving 5 chunks instead of 3 picks up the detail chunk.
- **Medium chunks (128 tokens) with overlap (25)** tend to keep the full Recharge Days section together in one chunk, so even top_k=3 gets 4/4 facts.
- **Brute force (top_k=10)** gets all facts but scores lower because efficiency is penalized.

The sweet spot: **enough chunk size to keep related information together, with just enough retrieval to cover the answer.** This is the fundamental chunking tradeoff.

---

## 10. The Plot Twist: One Size Does NOT Fit All

You just optimized chunking for the Recharge Days question. Now let's see what happens when we use those same "winning" parameters on a **different type of question**.

The question: *"How much was approved for the Project Meridian investment and what is the budget breakdown?"*

The answer lives in `board_meeting_q4_2024.md` — a bulleted budget list ($850K personnel, $600K infrastructure, $350K training, $300K consulting). This is **structurally different** from the Recharge Days paragraph.

Run the next cell and compare what happens.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# PLOT TWIST: Project Meridian — different content, different result
# ═══════════════════════════════════════════════════════════════════════

MERIDIAN_QUESTION = "How much was approved for the Project Meridian investment and what is the budget breakdown?"
MERIDIAN_DOCS = [
    "board_meeting_q4_2024.md",
    "memo_ceo_annual_kickoff.md",
    "financial_report_q4_2024.md",
    "engineering_standup_2025_01.md",
    "board_meeting_q3_2024.md",
]

MERIDIAN_FACTS = {
    "Total ($2.1M)": ["$2.1 million", "$2.1m", "2.1 million"],
    "Personnel ($850K)": ["$850", "850k", "personnel"],
    "Infrastructure ($600K)": ["$600", "600k", "infrastructure"],
    "Training ($350K)": ["$350", "350k", "training", "upskilling"],
}

data_dir = _root / "data" / "northbrook"

# Run the Meridian question at TWO different chunk sizes
for label, cs, ov, tk in [
    ("Your Recharge Days winner", 128, 25, 3),
    ("Tiny chunks", 50, 0, 3),
]:
    all_chunks = []
    for fname in MERIDIAN_DOCS:
        text = (data_dir / fname).read_text()
        all_chunks.extend(chunk_document(text, source=fname, chunk_size=cs, overlap=ov))

    client = chromadb.PersistentClient(path=CHROMA_PATH)
    try:
        client.delete_collection(WORKBENCH_COLLECTION)
    except Exception:
        pass
    ingest_chunks(all_chunks, collection_name=WORKBENCH_COLLECTION)

    results = verify_ingestion(MERIDIAN_QUESTION, n_results=tk, collection_name=WORKBENCH_COLLECTION)
    retrieved = []
    for i, d in enumerate(results["distances"][0]):
        retrieved.append({
            "text": results["documents"][0][i],
            "metadata": results["metadatas"][0][i],
            "score": round(1 - d, 4),
        })

    # Call Claude
    blocks = []
    for c in retrieved:
        src = c["metadata"].get("source", "Unknown")
        blocks.append(f"[Source: {src}, Score: {c['score']:.3f}]\n{c['text']}")
    context = "\n\n---\n\n".join(blocks)
    user_msg = f"Context:\n\n{context}\n\n---\n\nQuestion: {MERIDIAN_QUESTION}"
    result = call_claude_with_usage(
        prompt=user_msg, system_prompt=WORKBENCH_SYSTEM_PROMPT, temperature=0.0,
        model="claude-haiku-4-5",
    )

    # Check facts
    answer_lower = result["text"].lower()
    facts = {name: any(kw.lower() in answer_lower for kw in kws)
             for name, kws in MERIDIAN_FACTS.items()}
    hits = sum(facts.values())

    print(f"\n{'='*60}")
    print(f"  {label} (size={cs}, overlap={ov}, top_k={tk})")
    print(f"  Facts: {hits}/4")
    for name, hit in facts.items():
        print(f"    {'✓' if hit else '✗'} {name}")
    print(f"  Chunks: {len(all_chunks)} total, {len(retrieved)} retrieved")
    print(f"  Tokens: {result['input_tokens']}")
    print(f"{'='*60}")

# Cleanup
try:
    client.delete_collection(WORKBENCH_COLLECTION)
except Exception:
    pass

### The Lesson

The parameters that scored 91 on Recharge Days may score differently on Project Meridian. Why?

- **Recharge Days** is a single continuous paragraph — medium chunks (500) keep it whole.
- **Project Meridian's budget** is a bulleted list — each line item is short. Tiny chunks (200) can capture individual items, and they each retrieve well because they're specific.

**There is no universal "best" chunk size.** The right parameters depend on:
- The **structure** of your documents (paragraphs vs lists vs tables)
- The **type of questions** users will ask (broad vs specific)
- The **density** of information (one answer per paragraph vs scattered across sections)

This is why Session 3.2 introduces **metadata-filtered retrieval** — instead of relying solely on chunk size, you can use document type, source, and other metadata to narrow retrieval before similarity search even runs.

> *"If your chunks are bad, no amount of prompt engineering will save you."*

---

## 11. Session Recap

### Where we are in the pipeline

| Session | What We Built | Status |
|---------|--------------|--------|
| **1.1** | API connection + generation | Done |
| **1.2** | Structured extraction with Pydantic | Done |
| **2.1** | Embeddings + similarity measurement | Done |
| **2.2** | Chunking + vector store ingestion | **Done (today)** |
| **3.1** | Naive RAG — retrieve + generate | Next session |
| **3.2** | Metadata-filtered RAG + evaluation | Coming |

### What we built today

1. **Chunking exploration** — compared fixed-size vs paragraph-aware strategies
2. **Chunk destruction demo** — saw how bad chunking destroys meaning at boundaries
3. **Vector store ingestion** — embedded and stored the full Northbrook corpus in ChromaDB
4. **The Chunking Workbench** — proved that chunk_size, overlap, and top_k directly change answer quality
5. **The Plot Twist** — discovered that optimal parameters differ across document types and question types

### Key takeaways

- **Chunking is a design decision** that directly impacts downstream retrieval quality
- **Overlap is insurance, not a solution** — it recovers some context at seams but costs more
- **There is no universal "best" chunk size** — it depends on document structure and query type
- **Metadata enables smarter retrieval** — filtering by doc_type narrows the search space (Week 3)
- **Brute force works but wastes resources** — the engineering challenge is precision, not volume

### Lab 1 — Assigned Today

Lab 1 (Batch Extraction Pipeline) brings together Weeks 1 and 2:

| Component | Source |
|-----------|--------|
| Pydantic schema design | Session 1.2 |
| Batch extraction with error handling | Session 1.2 |
| Chunking with metadata | Session 2.2 (today) |
| Output ready for ingestion | Session 2.2 (today) |

**Due:** End of Session 3.2 (two weeks) | **Weight:** 20% of course grade